# EDA - Zalo AI Traffic Sign Dataset
Notebook này thực thi toàn bộ Kế hoạch EDA từ E0 đến E6 được thiết kế trong `eda_KeHoach_TrienKhai.md`.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import cv2
import os
import random
from matplotlib.patches import Rectangle
from sklearn.cluster import KMeans
import squarify # Cần cài đặt: pip install squarify

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN TỚI THƯ MỤC DATA
# ==========================================
DATA_DIR = r"D:\Học thống kê\Final Project\data"
JSON_PATH = os.path.join(DATA_DIR, "train_traffic_sign_dataset.json")
IMAGES_DIR = os.path.join(DATA_DIR, "images")
REPORTS_DIR = r"D:\Học thống kê\Final Project\Traffic-Sign-Detection-ZaloAI\reports\charts"
os.makedirs(REPORTS_DIR, exist_ok=True)

with open(JSON_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

annotations = data['annotations']
images_info = data['images']
categories = {cat['id']: cat['name'] for cat in data['categories']}

df_anno = pd.DataFrame(annotations)
df_img = pd.DataFrame(images_info)
print(f"Tổng số ảnh: {len(df_img)}")
print(f"Tổng số biển báo: {len(df_anno)}")


## [E0] Trực quan hóa Dữ liệu mẫu (Visual Inspection)
Vẽ Bounding Box lên một vài ảnh ngẫu nhiên để kiểm tra tính chính xác của nhãn.

In [ ]:
import os
import matplotlib.pyplot as plt
import cv2
import numpy as np
from matplotlib.patches import Rectangle
import random

available_images = set(os.listdir(IMAGES_DIR))
valid_df_img = df_img[df_img['file_name'].isin(available_images)]
valid_ids = valid_df_img['id'].unique()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

if len(valid_ids) == 0:
    fig.text(0.5, 0.5, f"LỖI: {len(available_images)} ảnh bạn tải về không nằm trong tập Train!\nHãy lên Kaggle tải đúng ảnh trong thư mục 'traffic_train/images/'", ha='center', va='center', fontsize=20, color='red', fontweight='bold')
    for i in range(4):
        axes[i].axis('off')
else:
    if len(valid_ids) >= 4:
        sample_image_ids = random.sample(list(valid_ids), 4)
    else:
        sample_image_ids = list(valid_ids)

    for i in range(4):
        if i < len(sample_image_ids):
            img_id = sample_image_ids[i]
            img_data = valid_df_img[valid_df_img['id'] == img_id].iloc[0]
            img_name = img_data['file_name']
            img_path = os.path.join(IMAGES_DIR, img_name)
            
            img_array = np.fromfile(img_path, np.uint8)
            img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[i].imshow(img)
            
            annos = df_anno[df_anno['image_id'] == img_id]
            found_classes = []
            for _, anno in annos.iterrows():
                x, y, w, h = anno['bbox']
                cat_name = categories[anno['category_id']]
                found_classes.append(cat_name)
                
                # CHỈ vẽ khung màu đỏ, KHÔNG vẽ chữ đè lên ảnh
                rect = Rectangle((x, y), w, h, linewidth=2, edgecolor='red', facecolor='none')
                axes[i].add_patch(rect)
            
            # Ghi danh sách các biển báo xuống dưới bức ảnh (làm tiêu đề phụ)
            class_text = ", ".join(set(found_classes))
            axes[i].set_title(f"Ảnh: {img_name}\nBiển báo: {class_text}", fontsize=14, fontweight='bold', color='darkred')
            axes[i].axis('off')
        else:
            axes[i].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'E0_visual_inspection.png'), bbox_inches='tight', dpi=300)
plt.show()

print("-> KẾT LUẬN E0: Đã vẽ khung Bounding Box. Việc đưa nhãn (Label) ra ngoài ảnh giúp không che khuất biển báo siêu nhỏ.")


## [E1] Tổng quan Phân bố nhãn (Class Balance Dashboard)

In [ ]:
# 1. Total Objects
obj_counts = df_anno['category_id'].value_counts()
# 2. Total Images containing class
img_counts = df_anno.groupby('category_id')['image_id'].nunique()
# 3. Avg count on image
avg_counts = obj_counts / img_counts
# 4. Avg Area on image
df_anno['area'] = df_anno['bbox'].apply(lambda x: x[2] * x[3])
df_merged_e1 = df_anno.merge(df_img, left_on='image_id', right_on='id')
df_merged_e1['img_area'] = df_merged_e1['width'] * df_merged_e1['height']
df_merged_e1['area_ratio'] = (df_merged_e1['area'] / df_merged_e1['img_area']) * 100
avg_area = df_merged_e1.groupby('category_id')['area_ratio'].mean()

# Combine into a single DataFrame
e1_df = pd.DataFrame({
    'Class': [categories[c] for c in obj_counts.index],
    'Objects': obj_counts.values,
    'Images': img_counts.loc[obj_counts.index].values,
    'Avg Count/Img': avg_counts.loc[obj_counts.index].round(2).values,
    'Avg Area (%)': avg_area.loc[obj_counts.index].round(2).values
})

fig = plt.figure(figsize=(18, 8))
ax1 = plt.subplot2grid((1, 3), (0, 0), colspan=2)
ax2 = plt.subplot2grid((1, 3), (0, 2))

# Bar chart
sns.barplot(data=e1_df, x='Objects', y='Class', palette='Set2', ax=ax1)
ax1.set_title('Phân bố số lượng Biển báo (Objects)', fontsize=14)
for i, v in enumerate(e1_df['Objects']):
    ax1.text(v + 10, i, str(v), va='center', fontweight='bold')

# Table
ax2.axis('off')
table = ax2.table(cellText=e1_df.values, colLabels=e1_df.columns, cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 2)

plt.suptitle('E1: Bảng điều khiển Tổng quan Phân bố Nhãn (Class Balance Dashboard)', fontsize=18, fontweight='bold')
plt.savefig(os.path.join(REPORTS_DIR, 'E1_class_balance.png'), bbox_inches='tight', dpi=300)
plt.show()

print("\n--- BẢNG THỐNG KÊ CHI TIẾT ---")
print(e1_df.to_string(index=False))
print("\n-> KẾT LUẬN M1.1: Dữ liệu mất cân bằng nặng. BẮT BUỘC dùng Focal Loss hoặc Class Weights.")


## [E2] Phân bố Mật độ vật thể (Object Distribution Heatmap)

In [ ]:
obj_per_img = df_anno.groupby(['image_id', 'category_id']).size().unstack(fill_value=0)
obj_per_img.columns = [categories[c] for c in obj_per_img.columns]

density_matrix = {}
for col in obj_per_img.columns:
    counts = obj_per_img[obj_per_img[col] > 0][col].value_counts().sort_index()
    density_matrix[col] = counts

density_df = pd.DataFrame(density_matrix).fillna(0).T

plt.figure(figsize=(10, 6))
sns.heatmap(density_df, annot=True, fmt='g', cmap='YlOrRd')
plt.title('E2: Số lượng ảnh chứa 1, 2, 3... biển báo (Object Density)', fontsize=14)
plt.xlabel('Số lượng biển báo xuất hiện trong 1 ảnh')
plt.ylabel('Loại biển báo')
plt.savefig(os.path.join(REPORTS_DIR, 'E2_object_density.png'), bbox_inches='tight')
plt.show()

print("-> KẾT LUẬN M3.3: Có rất nhiều ảnh chứa 2, 3, 4 biển báo cùng loại xếp cạnh nhau. Cần tăng IoU threshold của NMS.")


## [E3] Kích thước chi tiết & Tree Map (Class Sizes & Tree Map)

In [ ]:
df_merged_e3 = df_merged_e1.copy()
df_merged_e3['w_ratio'] = (df_merged_e3['bbox'].apply(lambda x: x[2]) / df_merged_e3['width']) * 100
df_merged_e3['h_ratio'] = (df_merged_e3['bbox'].apply(lambda x: x[3]) / df_merged_e3['height']) * 100

stats = df_merged_e3.groupby('category_id').agg(
    Min_W=('w_ratio', 'min'), Max_W=('w_ratio', 'max'), Avg_W=('w_ratio', 'mean'),
    Min_H=('h_ratio', 'min'), Max_H=('h_ratio', 'max'), Avg_H=('h_ratio', 'mean'),
    Avg_Area=('area_ratio', 'mean')
).reset_index()
stats['Class'] = stats['category_id'].map(categories)

for col in stats.columns:
    if col not in ['category_id', 'Class']:
        stats[col] = stats[col].round(2)

print("--- BẢNG THỐNG KÊ KÍCH THƯỚC (CLASS SIZES) ---")
display_cols = ['Class', 'Min_W', 'Max_W', 'Avg_W', 'Min_H', 'Max_H', 'Avg_H', 'Avg_Area']
print(stats[display_cols].to_string(index=False))
print("\n")

plt.figure(figsize=(14, 8))
labels = [f"{row['Class']}\n{row['Avg_Area']}%" for idx, row in stats.iterrows()]
sizes = stats['Avg_Area'].values
colors = sns.color_palette('pastel')[0:len(labels)]

try:
    squarify.plot(sizes=sizes, label=labels, color=colors, alpha=0.8, edgecolor='white', linewidth=2, text_kwargs={'fontsize':12, 'weight':'bold'})
    plt.title('E3: Tree Map - Tỷ lệ diện tích trung bình của biển báo so với ảnh gốc', fontsize=18, fontweight='bold', pad=20)
    plt.axis('off')
    plt.savefig(os.path.join(REPORTS_DIR, 'E3_treemap_sizes.png'), bbox_inches='tight', dpi=300)
    plt.show()
except NameError:
    print("Vui lòng cài đặt squarify: pip install squarify")

print("-> KẾT LUẬN M2.1 & M2.3: Diện tích trung bình chỉ < 1%. Chắc chắn là Small Object Detection. Cần SAHI và P2 Layer.")


## [E4] Bản đồ nhiệt Không gian (Spatial Heatmap)

In [ ]:
df_merged = df_anno.merge(df_img, left_on='image_id', right_on='id')
df_merged['x_center_norm'] = (df_merged['bbox'].apply(lambda x: x[0] + x[2]/2)) / df_merged['width']
df_merged['y_center_norm'] = (df_merged['bbox'].apply(lambda x: x[1] + x[3]/2)) / df_merged['height']

plt.figure(figsize=(10, 6))
sns.kdeplot(
    data=df_merged, x="x_center_norm", y="y_center_norm", 
    fill=True, cmap="mako", thresh=0, levels=100
)
plt.gca().invert_yaxis()
plt.title('E4: Bản đồ nhiệt vị trí phân bố của biển báo trên ảnh (Spatial Heatmap)', fontsize=16)
plt.xlim(0, 1)
plt.ylim(1, 0)
plt.savefig(os.path.join(REPORTS_DIR, 'E4_spatial_heatmap.png'), bbox_inches='tight')
plt.show()

print("-> KẾT LUẬN M3.2: Biển báo tập trung cực dày đặc ở LỀ PHẢI. Bắt buộc dùng BBox-Safe Augmentation (min_visibility=0.5).")


## [E5] Ma trận Đồng xuất hiện (Co-occurrence Matrix)

In [ ]:
co_matrix = pd.DataFrame(0, index=categories.values(), columns=categories.values())

for img_id, group in df_anno.groupby('image_id'):
    cats = group['category_id'].map(categories).unique()
    for i in range(len(cats)):
        co_matrix.loc[cats[i], cats[i]] += 1
        for j in range(i+1, len(cats)):
            co_matrix.loc[cats[i], cats[j]] += 1
            co_matrix.loc[cats[j], cats[i]] += 1

plt.figure(figsize=(10, 8))
sns.heatmap(co_matrix, annot=True, fmt='d', cmap='crest', linewidths=1, annot_kws={'size': 12, 'weight': 'bold'})
plt.title('E5: Ma trận Đồng xuất hiện (Co-occurrence Matrix)', fontsize=18, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=12, fontweight='bold')
plt.yticks(rotation=0, fontsize=12, fontweight='bold')
plt.savefig(os.path.join(REPORTS_DIR, 'E5_cooccurrence.png'), bbox_inches='tight', dpi=300)
plt.show()


## [E6] Phổ phân bố Tỷ lệ khung hình & K-Means Anchors

In [ ]:
df_anno['w'] = df_anno['bbox'].apply(lambda x: x[2])
df_anno['h'] = df_anno['bbox'].apply(lambda x: x[3])
df_anno['aspect_ratio'] = df_anno['w'] / df_anno['h']

plt.figure(figsize=(10, 6))
sns.histplot(df_anno['aspect_ratio'], bins=50, color='purple', kde=True)
plt.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Tỷ lệ 1:1 (Vuông/Tròn)')
plt.title('E6: Phân bố Tỷ lệ khung hình (Aspect Ratio)', fontsize=16)
plt.xlabel('Aspect Ratio (Width / Height)')
plt.legend()
plt.xlim(0, 3)
plt.savefig(os.path.join(REPORTS_DIR, 'E6_aspect_ratio.png'), bbox_inches='tight')
plt.show()

print("\n[M3.1] CHẠY THUẬT TOÁN K-MEANS TÌM ANCHOR BOX CHO FASTER R-CNN")
boxes_wh = df_anno[['w', 'h']].values
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans.fit(boxes_wh)
anchors = kmeans.cluster_centers_
anchors = anchors[np.argsort(anchors[:, 0] * anchors[:, 1])]

print("5 Kích thước Anchor Box chuẩn (W x H) đo ni đóng giày cho Zalo AI:")
for i, (w, h) in enumerate(anchors):
    print(f"  Anchor {i+1}: {w:.1f} x {h:.1f} (Aspect Ratio = {w/h:.2f})")
print("\n-> Hãy copy các thông số này nạp vào AnchorGenerator của Faster R-CNN!")
